# Step 8: Ensemble Voting Classifier

## 🎯 Objective:
Combine multiple models to create a **more robust and accurate** ensemble classifier.

## 💡 Why Ensemble?

### **"Wisdom of the Crowd"**
```
Single Model:
- XGBoost might be good at trend detection
- LightGBM might be good at volatility patterns
- CatBoost might be good at correlation patterns
- Each has strengths and weaknesses

Ensemble:
- Combine all three models
- Average their predictions (voting)
- Reduce individual model errors
- More stable and robust
```

### **Types of Voting:**
```
1. Hard Voting:
   - Each model votes 0 or 1
   - Majority wins
   - Example: XGB=1, LGBM=1, Cat=0 → Final=1

2. Soft Voting (BETTER!):
   - Each model gives probability
   - Average probabilities
   - Example: XGB=0.7, LGBM=0.8, Cat=0.6 → Avg=0.7 → Final=1
   - More nuanced, usually better performance
```

## 📊 Expected Improvement:
```
Best Single Model (Step 7): 68-72%
Ensemble (Step 8):          70-75%
Improvement:                +2-5%
```

## 🎓 Ensemble Strategies:
1. **Equal Weight**: All models contribute equally
2. **Weighted**: Better models get higher weight
3. **Stacking**: Use meta-learner (advanced)

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score, roc_curve
)

# Gradient Boosting
import xgboost as xgb
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Libraries loaded successfully!")
print("\n📦 Step 8: Ensemble Voting Classifier")
print("  Strategy: Combine XGBoost + LightGBM + CatBoost")
print("  Method: Soft voting (probability averaging)")

✅ Libraries loaded successfully!

📦 Step 8: Ensemble Voting Classifier
  Strategy: Combine XGBoost + LightGBM + CatBoost
  Method: Soft voting (probability averaging)


## 1. Data Loading & Feature Engineering (Same as Step 7)

In [2]:
# Load data
file_path = 'dataset_2023_2025.xlsx'
data = pd.read_excel(file_path, index_col=0, parse_dates=True)

print(f"📊 Dataset loaded: {len(data)} rows")
print(f"📅 Date range: {data.index[0].date()} to {data.index[-1].date()}")

# Technical indicator functions (same as Step 7)
def calculate_rsi(prices, period=14):
    delta = prices.diff()
    gain = delta.where(delta > 0, 0).rolling(window=period).mean()
    loss = -delta.where(delta < 0, 0).rolling(window=period).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

def calculate_macd(prices, fast=12, slow=26, signal=9):
    ema_fast = prices.ewm(span=fast, adjust=False).mean()
    ema_slow = prices.ewm(span=slow, adjust=False).mean()
    macd = ema_fast - ema_slow
    macd_signal = macd.ewm(span=signal, adjust=False).mean()
    macd_histogram = macd - macd_signal
    return macd, macd_signal, macd_histogram

def calculate_bollinger_bands(prices, period=20, num_std=2):
    sma = prices.rolling(window=period).mean()
    std = prices.rolling(window=period).std()
    upper_band = sma + (num_std * std)
    lower_band = sma - (num_std * std)
    bb_position = (prices - lower_band) / (upper_band - lower_band)
    bb_width = (upper_band - lower_band) / sma
    return bb_position, bb_width

def calculate_atr(prices, period=14):
    returns = prices.pct_change().abs()
    return returns.rolling(window=period).mean()

def rolling_avg_correlation(returns, window=20):
    corr_values = []
    for i in range(len(returns)):
        if i < window:
            corr_values.append(np.nan)
        else:
            window_data = returns.iloc[i-window:i]
            corr_matrix = window_data.corr()
            n = len(corr_matrix)
            avg_corr = (corr_matrix.sum().sum() - n) / (n * n - n)
            corr_values.append(avg_corr)
    return pd.Series(corr_values, index=returns.index)

print("✅ Helper functions defined")

📊 Dataset loaded: 990 rows
📅 Date range: 2023-04-17 to 2025-12-31
✅ Helper functions defined


In [3]:
# Feature engineering (same as Step 7)
btc_prices = data['BTC-USD']
btc_returns = btc_prices.pct_change()
returns = data.pct_change()
market_return = returns.mean(axis=1)

features = pd.DataFrame(index=data.index)

print("🔄 Creating features...")

# BTC features
features['BTC_Mom_5'] = btc_returns.rolling(5).mean()
features['BTC_Mom_10'] = btc_returns.rolling(10).mean()
features['BTC_Mom_20'] = btc_returns.rolling(20).mean()
features['BTC_Mom_50'] = btc_returns.rolling(50).mean()
features['BTC_Vol_5'] = btc_returns.rolling(5).std()
features['BTC_Vol_20'] = btc_returns.rolling(20).std()

# Technical indicators
features['RSI_14'] = calculate_rsi(btc_prices, 14)
macd, macd_signal, macd_hist = calculate_macd(btc_prices)
features['MACD'] = macd
features['MACD_Signal'] = macd_signal
features['MACD_Hist'] = macd_hist
bb_pos, bb_width = calculate_bollinger_bands(btc_prices, 20)
features['BB_Position'] = bb_pos
features['BB_Width'] = bb_width
features['ATR_14'] = calculate_atr(btc_prices, 14)

# Market features
features['Market_Vol_20'] = market_return.rolling(20).std()
features['Avg_Correlation'] = rolling_avg_correlation(returns, 20)

# Magnitude-aware target
THRESHOLD = 0.005
next_day_return = btc_returns.shift(-1)
features['Target'] = np.where(
    next_day_return > THRESHOLD, 1,
    np.where(next_day_return < -THRESHOLD, 0, np.nan)
)

features_clean = features.dropna()

print(f"✅ Features created: {len(features_clean.columns)-1} features")
print(f"   Training samples: {len(features_clean)}")

🔄 Creating features...
✅ Features created: 15 features
   Training samples: 685


## 2. Train/Test Split

In [4]:
# Feature columns
feature_cols = [col for col in features_clean.columns if col != 'Target']

# Split
train_mask = (features_clean.index.year >= 2023) & (features_clean.index.year <= 2024)
test_mask = (features_clean.index.year == 2025)

X_train = features_clean.loc[train_mask, feature_cols]
y_train = features_clean.loc[train_mask, 'Target']
X_test = features_clean.loc[test_mask, feature_cols]
y_test = features_clean.loc[test_mask, 'Target']

print("=" * 60)
print("TRAIN/TEST SPLIT")
print("=" * 60)
print(f"\n📚 Training: {len(X_train)} samples")
print(f"🧪 Testing: {len(X_test)} samples")
print("=" * 60)

TRAIN/TEST SPLIT

📚 Training: 418 samples
🧪 Testing: 267 samples


## 3. Train Individual Models (Baseline)

In [5]:
print("🔄 Training individual models...\n")

# XGBoost
print("1️⃣ XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='logloss',
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train, verbose=False)
xgb_pred = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_acc = accuracy_score(y_test, xgb_pred)
print(f"   Accuracy: {xgb_acc:.2%}")

# LightGBM
print("\n2️⃣ LightGBM...")
lgbm_model = LGBMClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1
)
lgbm_model.fit(X_train, y_train)
lgbm_pred = lgbm_model.predict(X_test)
lgbm_proba = lgbm_model.predict_proba(X_test)[:, 1]
lgbm_acc = accuracy_score(y_test, lgbm_pred)
print(f"   Accuracy: {lgbm_acc:.2%}")

# CatBoost
print("\n3️⃣ CatBoost...")
catboost_model = CatBoostClassifier(
    iterations=200,
    depth=5,
    learning_rate=0.1,
    random_state=42,
    verbose=False
)
catboost_model.fit(X_train, y_train)
catboost_pred = catboost_model.predict(X_test)
catboost_proba = catboost_model.predict_proba(X_test)[:, 1]
catboost_acc = accuracy_score(y_test, catboost_pred)
print(f"   Accuracy: {catboost_acc:.2%}")

print("\n✅ Individual models trained")

🔄 Training individual models...

1️⃣ XGBoost...
   Accuracy: 50.94%

2️⃣ LightGBM...
   Accuracy: 53.56%

3️⃣ CatBoost...
   Accuracy: 51.31%

✅ Individual models trained


## 4. Create Ensemble: Equal Weight Voting

In [6]:
print("\n🔄 Creating Ensemble (Equal Weight)...\n")

# Create voting classifier with equal weights
ensemble_equal = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('catboost', catboost_model)
    ],
    voting='soft',  # Use probabilities
    weights=None    # Equal weights
)

# Note: Models already trained, just need to fit ensemble wrapper
ensemble_equal.fit(X_train, y_train)

# Predictions
ensemble_equal_pred = ensemble_equal.predict(X_test)
ensemble_equal_proba = ensemble_equal.predict_proba(X_test)[:, 1]
ensemble_equal_acc = accuracy_score(y_test, ensemble_equal_pred)

print(f"✅ Ensemble (Equal Weight) Accuracy: {ensemble_equal_acc:.2%}")
print(f"\n📊 Comparison:")
print(f"   XGBoost:     {xgb_acc:.2%}")
print(f"   LightGBM:    {lgbm_acc:.2%}")
print(f"   CatBoost:    {catboost_acc:.2%}")
print(f"   Ensemble:    {ensemble_equal_acc:.2%}")

best_single = max(xgb_acc, lgbm_acc, catboost_acc)
improvement = (ensemble_equal_acc - best_single) * 100
print(f"\n   Improvement vs best single: {improvement:+.2f}%")


🔄 Creating Ensemble (Equal Weight)...

✅ Ensemble (Equal Weight) Accuracy: 52.43%

📊 Comparison:
   XGBoost:     50.94%
   LightGBM:    53.56%
   CatBoost:    51.31%
   Ensemble:    52.43%

   Improvement vs best single: -1.12%


## 5. Create Ensemble: Weighted Voting

In [7]:
print("\n🔄 Creating Ensemble (Weighted by Performance)...\n")

# Calculate weights based on individual accuracy
# Better models get higher weight
total_acc = xgb_acc + lgbm_acc + catboost_acc
weight_xgb = xgb_acc / total_acc
weight_lgbm = lgbm_acc / total_acc
weight_catboost = catboost_acc / total_acc

# Normalize to sum to 3 (sklearn convention)
weights = [weight_xgb * 3, weight_lgbm * 3, weight_catboost * 3]

print(f"📊 Calculated Weights:")
print(f"   XGBoost:  {weights[0]:.3f} (based on {xgb_acc:.2%} accuracy)")
print(f"   LightGBM: {weights[1]:.3f} (based on {lgbm_acc:.2%} accuracy)")
print(f"   CatBoost: {weights[2]:.3f} (based on {catboost_acc:.2%} accuracy)")

# Create weighted ensemble
ensemble_weighted = VotingClassifier(
    estimators=[
        ('xgb', xgb_model),
        ('lgbm', lgbm_model),
        ('catboost', catboost_model)
    ],
    voting='soft',
    weights=weights
)

ensemble_weighted.fit(X_train, y_train)

# Predictions
ensemble_weighted_pred = ensemble_weighted.predict(X_test)
ensemble_weighted_proba = ensemble_weighted.predict_proba(X_test)[:, 1]
ensemble_weighted_acc = accuracy_score(y_test, ensemble_weighted_pred)

print(f"\n✅ Ensemble (Weighted) Accuracy: {ensemble_weighted_acc:.2%}")
print(f"\n📊 All Results:")
print(f"   Best Single:      {best_single:.2%}")
print(f"   Ensemble (Equal): {ensemble_equal_acc:.2%}")
print(f"   Ensemble (Weighted): {ensemble_weighted_acc:.2%}")


🔄 Creating Ensemble (Weighted by Performance)...

📊 Calculated Weights:
   XGBoost:  0.981 (based on 50.94% accuracy)
   LightGBM: 1.031 (based on 53.56% accuracy)
   CatBoost: 0.988 (based on 51.31% accuracy)

✅ Ensemble (Weighted) Accuracy: 52.43%

📊 All Results:
   Best Single:      53.56%
   Ensemble (Equal): 52.43%
   Ensemble (Weighted): 52.43%


## 6. Manual Ensemble: Custom Probability Averaging

In [8]:
print("\n🔄 Creating Manual Ensemble (Custom Averaging)...\n")

# Strategy 1: Simple average
avg_proba_simple = (xgb_proba + lgbm_proba + catboost_proba) / 3
manual_pred_simple = (avg_proba_simple > 0.5).astype(int)
manual_acc_simple = accuracy_score(y_test, manual_pred_simple)

print(f"1️⃣ Simple Average: {manual_acc_simple:.2%}")

# Strategy 2: Weighted average (performance-based)
avg_proba_weighted = (
    xgb_proba * xgb_acc + 
    lgbm_proba * lgbm_acc + 
    catboost_proba * catboost_acc
) / total_acc
manual_pred_weighted = (avg_proba_weighted > 0.5).astype(int)
manual_acc_weighted = accuracy_score(y_test, manual_pred_weighted)

print(f"2️⃣ Weighted Average: {manual_acc_weighted:.2%}")

# Strategy 3: Majority voting (hard voting)
votes = np.array([xgb_pred, lgbm_pred, catboost_pred])
majority_pred = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=0, arr=votes)
majority_acc = accuracy_score(y_test, majority_pred)

print(f"3️⃣ Majority Voting: {majority_acc:.2%}")

# Strategy 4: Confidence-based (only predict if all agree or high confidence)
# If all models agree, use that. Otherwise, use weighted average
all_agree = (xgb_pred == lgbm_pred) & (lgbm_pred == catboost_pred)
confident_pred = np.where(
    all_agree,
    xgb_pred,  # All agree, use prediction
    manual_pred_weighted  # Disagree, use weighted average
)
confident_acc = accuracy_score(y_test, confident_pred)

print(f"4️⃣ Confidence-Based: {confident_acc:.2%}")
print(f"   (Agreement rate: {all_agree.sum() / len(all_agree) * 100:.1f}%)")


🔄 Creating Manual Ensemble (Custom Averaging)...

1️⃣ Simple Average: 52.43%
2️⃣ Weighted Average: 52.43%


TypeError: Cannot cast array data from dtype('float64') to dtype('int64') according to the rule 'safe'

## 7. Comprehensive Performance Comparison

In [ ]:
# Compile all results
results_all = [
    {
        'Model': 'XGBoost',
        'Type': 'Single',
        'Accuracy': xgb_acc,
        'Precision': precision_score(y_test, xgb_pred, zero_division=0),
        'Recall': recall_score(y_test, xgb_pred, zero_division=0),
        'F1-Score': f1_score(y_test, xgb_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, xgb_proba)
    },
    {
        'Model': 'LightGBM',
        'Type': 'Single',
        'Accuracy': lgbm_acc,
        'Precision': precision_score(y_test, lgbm_pred, zero_division=0),
        'Recall': recall_score(y_test, lgbm_pred, zero_division=0),
        'F1-Score': f1_score(y_test, lgbm_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, lgbm_proba)
    },
    {
        'Model': 'CatBoost',
        'Type': 'Single',
        'Accuracy': catboost_acc,
        'Precision': precision_score(y_test, catboost_pred, zero_division=0),
        'Recall': recall_score(y_test, catboost_pred, zero_division=0),
        'F1-Score': f1_score(y_test, catboost_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, catboost_proba)
    },
    {
        'Model': 'Ensemble (Equal)',
        'Type': 'Ensemble',
        'Accuracy': ensemble_equal_acc,
        'Precision': precision_score(y_test, ensemble_equal_pred, zero_division=0),
        'Recall': recall_score(y_test, ensemble_equal_pred, zero_division=0),
        'F1-Score': f1_score(y_test, ensemble_equal_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, ensemble_equal_proba)
    },
    {
        'Model': 'Ensemble (Weighted)',
        'Type': 'Ensemble',
        'Accuracy': ensemble_weighted_acc,
        'Precision': precision_score(y_test, ensemble_weighted_pred, zero_division=0),
        'Recall': recall_score(y_test, ensemble_weighted_pred, zero_division=0),
        'F1-Score': f1_score(y_test, ensemble_weighted_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, ensemble_weighted_proba)
    },
    {
        'Model': 'Manual (Simple Avg)',
        'Type': 'Manual',
        'Accuracy': manual_acc_simple,
        'Precision': precision_score(y_test, manual_pred_simple, zero_division=0),
        'Recall': recall_score(y_test, manual_pred_simple, zero_division=0),
        'F1-Score': f1_score(y_test, manual_pred_simple, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, avg_proba_simple)
    },
    {
        'Model': 'Manual (Weighted Avg)',
        'Type': 'Manual',
        'Accuracy': manual_acc_weighted,
        'Precision': precision_score(y_test, manual_pred_weighted, zero_division=0),
        'Recall': recall_score(y_test, manual_pred_weighted, zero_division=0),
        'F1-Score': f1_score(y_test, manual_pred_weighted, zero_division=0),
        'ROC-AUC': roc_auc_score(y_test, avg_proba_weighted)
    },
    {
        'Model': 'Majority Voting',
        'Type': 'Manual',
        'Accuracy': majority_acc,
        'Precision': precision_score(y_test, majority_pred, zero_division=0),
        'Recall': recall_score(y_test, majority_pred, zero_division=0),
        'F1-Score': f1_score(y_test, majority_pred, zero_division=0),
        'ROC-AUC': np.nan  # No probability for hard voting
    }
]

results_df = pd.DataFrame(results_all)

print("\n" + "="*100)
print("STEP 8: COMPREHENSIVE ENSEMBLE COMPARISON")
print("="*100)
print(results_df.to_string(index=False))
print("="*100)

# Find best overall
best_idx = results_df['Accuracy'].idxmax()
best_result = results_df.iloc[best_idx]

print("\n🏆 BEST MODEL (Step 8):")
print(f"  Model: {best_result['Model']}")
print(f"  Type: {best_result['Type']}")
print(f"  Accuracy: {best_result['Accuracy']:.2%}")
print(f"  F1-Score: {best_result['F1-Score']:.4f}")
print(f"  ROC-AUC: {best_result['ROC-AUC']:.4f}")

print(f"\n📈 IMPROVEMENT:")
print(f"  Best Single Model: {best_single:.2%}")
print(f"  Best Ensemble: {best_result['Accuracy']:.2%}")
print(f"  Gain: {(best_result['Accuracy'] - best_single)*100:+.2f}%")

## 8. Visualization

In [ ]:
# Performance comparison chart
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Accuracy comparison
models = results_df['Model'].values
accuracies = results_df['Accuracy'].values
colors = ['skyblue', 'lightgreen', 'coral', 'gold', 'orange', 'pink', 'lightcyan', 'lavender']
x = np.arange(len(models))

bars = axes[0, 0].bar(x, accuracies, color=colors, edgecolor='black')
axes[0, 0].set_ylabel('Accuracy', fontsize=11)
axes[0, 0].set_title('Accuracy Comparison: Single vs Ensemble', fontsize=12, fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(models, rotation=45, ha='right', fontsize=9)
axes[0, 0].axhline(y=best_single, color='red', linestyle='--', alpha=0.5, label='Best Single')
axes[0, 0].grid(True, alpha=0.3, axis='y')
axes[0, 0].legend()
for bar in bars:
    height = bar.get_height()
    axes[0, 0].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2%}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# F1-Score comparison
f1_scores = results_df['F1-Score'].values
bars2 = axes[0, 1].bar(x, f1_scores, color=colors, edgecolor='black')
axes[0, 1].set_ylabel('F1-Score', fontsize=11)
axes[0, 1].set_title('F1-Score Comparison', fontsize=12, fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(models, rotation=45, ha='right', fontsize=9)
axes[0, 1].grid(True, alpha=0.3, axis='y')
for bar in bars2:
    height = bar.get_height()
    axes[0, 1].text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Grouped by type
type_groups = results_df.groupby('Type')['Accuracy'].mean()
axes[1, 0].bar(type_groups.index, type_groups.values, 
               color=['skyblue', 'gold', 'lightgreen'], edgecolor='black')
axes[1, 0].set_ylabel('Average Accuracy', fontsize=11)
axes[1, 0].set_title('Average Accuracy by Type', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3, axis='y')
for i, (typ, acc) in enumerate(type_groups.items()):
    axes[1, 0].text(i, acc, f'{acc:.2%}', ha='center', va='bottom', fontweight='bold')

# ROC Curves (only for models with probabilities)
prob_models = [
    ('XGBoost', xgb_proba, 'skyblue'),
    ('LightGBM', lgbm_proba, 'lightgreen'),
    ('CatBoost', catboost_proba, 'coral'),
    ('Ensemble (Equal)', ensemble_equal_proba, 'gold'),
    ('Ensemble (Weighted)', ensemble_weighted_proba, 'orange')
]

for name, proba, color in prob_models:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc_score = roc_auc_score(y_test, proba)
    linewidth = 2.5 if 'Ensemble' in name else 1.5
    axes[1, 1].plot(fpr, tpr, label=f'{name} (AUC={auc_score:.3f})', 
                    linewidth=linewidth, color=color)

axes[1, 1].plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.500)', linewidth=2)
axes[1, 1].set_xlabel('False Positive Rate', fontsize=11)
axes[1, 1].set_ylabel('True Positive Rate', fontsize=11)
axes[1, 1].set_title('ROC Curves', fontsize=12, fontweight='bold')
axes[1, 1].legend(loc='lower right', fontsize=8)
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle('Step 8: Ensemble Methods Performance', fontsize=15, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## 9. Confusion Matrix: Best Ensemble

In [ ]:
# Get predictions from best model
if best_result['Model'] == 'Ensemble (Equal)':
    best_pred = ensemble_equal_pred
elif best_result['Model'] == 'Ensemble (Weighted)':
    best_pred = ensemble_weighted_pred
elif best_result['Model'] == 'Manual (Simple Avg)':
    best_pred = manual_pred_simple
elif best_result['Model'] == 'Manual (Weighted Avg)':
    best_pred = manual_pred_weighted
elif best_result['Model'] == 'Majority Voting':
    best_pred = majority_pred
else:
    best_pred = xgb_pred if best_result['Model'] == 'XGBoost' else lgbm_pred

# Confusion matrix
cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='YlOrRd',
            xticklabels=['Bearish', 'Bullish'],
            yticklabels=['Bearish', 'Bullish'],
            cbar_kws={'label': 'Count'})
plt.title(f'Best Model Confusion Matrix\n{best_result["Model"]} - Accuracy: {best_result["Accuracy"]:.2%}',
          fontsize=13, fontweight='bold')
plt.ylabel('Actual', fontsize=11)
plt.xlabel('Predicted', fontsize=11)
plt.tight_layout()
plt.show()

print("\n📊 Classification Report (Best Model):")
print(classification_report(y_test, best_pred, target_names=['Bearish', 'Bullish']))

## 10. Summary & Recommendations

In [ ]:
print("\n" + "="*80)
print("STEP 8: ENSEMBLE SUMMARY")
print("="*80)

print("\n🎯 ENSEMBLE STRATEGIES TESTED:")
print("  1. Sklearn VotingClassifier (Equal Weight)")
print("  2. Sklearn VotingClassifier (Performance-Weighted)")
print("  3. Manual Simple Average")
print("  4. Manual Weighted Average")
print("  5. Majority Voting (Hard)")

print("\n🏆 BEST PERFORMER:")
print(f"  Model: {best_result['Model']}")
print(f"  Accuracy: {best_result['Accuracy']:.2%}")
print(f"  Precision: {best_result['Precision']:.2%}")
print(f"  Recall: {best_result['Recall']:.2%}")
print(f"  F1-Score: {best_result['F1-Score']:.4f}")

print("\n📊 PERFORMANCE SUMMARY:")
print(f"  Best Single Model: {best_single:.2%}")
print(f"  Best Ensemble: {best_result['Accuracy']:.2%}")
print(f"  Improvement: {(best_result['Accuracy'] - best_single)*100:+.2f}%")

# Average by type
print("\n📈 AVERAGE BY TYPE:")
for typ, group in results_df.groupby('Type'):
    avg_acc = group['Accuracy'].mean()
    print(f"  {typ}: {avg_acc:.2%}")

print("\n💡 KEY INSIGHTS:")
if best_result['Type'] == 'Ensemble' or best_result['Type'] == 'Manual':
    print("  ✅ Ensemble outperforms individual models")
    print("  ✅ Combining models reduces variance and improves stability")
else:
    print("  ⚠️ Single model still best (models may be too similar)")
    print("  💡 Consider diversifying model types or features")

final_acc = best_result['Accuracy']
if final_acc >= 0.70:
    print("\n🎉 EXCELLENT! Accuracy ≥ 70%")
    print("\n✅ READY FOR THESIS:")
    print("  - Use best ensemble in full strategy (Step 5)")
    print("  - Document ensemble methodology")
    print("  - Highlight improvement over baselines")
elif final_acc >= 0.65:
    print("\n✅ VERY GOOD! Accuracy ≥ 65%")
    print("\n📝 THESIS-READY:")
    print("  - Proceed to integration")
    print("  - Optional: Try Step 9 (hyperparameter tuning)")
elif final_acc >= 0.60:
    print("\n✅ ACCEPTABLE! Accuracy ≥ 60%")
    print("\n💡 NEXT STEPS:")
    print("  - Can proceed to thesis")
    print("  - Recommend Step 9 (hyperparameter tuning) for improvement")
else:
    print("\n⚠️ Below 60%")
    print("\n💡 RECOMMENDED:")
    print("  - Step 9: Hyperparameter tuning")
    print("  - Consider alternative approaches (regression, different features)")

print("\n" + "="*80)